# 03 — Model Training: LSTM Autoencoder

**Project:** robotic-bearing-pdm  
**Goal:** Train the LSTM Autoencoder on healthy bearing data, calibrate the anomaly
threshold, and visualise reconstruction error across the full 7-day run.

Pipeline:
```
data/processed/windows_train.npy  →  LSTM Autoencoder (trained on healthy only)
                                   →  Reconstruction error per window
                                   →  Threshold = μ + 3σ (fit on train errors)
                                   →  Anomaly flags on test set
                                   →  Save models/lstm_autoencoder.pt
```

> **Prerequisite:** Run `02_feature_engineering.ipynb` first to generate
> `data/processed/windows_train.npy`, `windows_test.npy`, and `scaler.json`.

## 0 · Setup

In [ ]:
import sys
from pathlib import Path

ROOT = Path().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from src.models.lstm_autoencoder import LSTMAutoencoder
from src.models.threshold import (
    collect_errors, compute_threshold, save_threshold, load_threshold
)

plt.rcParams["figure.dpi"] = 120
plt.rcParams["figure.figsize"] = (14, 4)

PROC_DIR  = ROOT / "data" / "processed"
MODEL_DIR = ROOT / "models"
MODEL_DIR.mkdir(exist_ok=True)

DEVICE    = "cuda" if torch.cuda.is_available() else "cpu"
SEQ_LEN   = 50

print(f"Device : {DEVICE}")
print(f"PyTorch: {torch.__version__}")

## 1 · Load Pre-computed Windows

In [ ]:
windows_train = np.load(PROC_DIR / "windows_train.npy")
windows_test  = np.load(PROC_DIR / "windows_test.npy")

with open(PROC_DIR / "feature_names.json") as f:
    feature_names = json.load(f)

n_features = windows_train.shape[2]

print(f"Train windows : {windows_train.shape}  (n_windows, seq_len, n_features)")
print(f"Test windows  : {windows_test.shape}")
print(f"n_features    : {n_features}")

## 2 · Hyperparameters

In [ ]:
CFG = dict(
    hidden_dim  = 64,
    latent_dim  = 32,
    n_layers    = 2,
    epochs      = 50,
    batch_size  = 64,
    lr          = 1e-3,
    k_sigma     = 3.0,    # threshold = μ + k·σ
)
print(CFG)

## 3 · Build Model & DataLoader

In [ ]:
model = LSTMAutoencoder(
    n_features = n_features,
    seq_len    = SEQ_LEN,
    hidden_dim = CFG["hidden_dim"],
    latent_dim = CFG["latent_dim"],
    n_layers   = CFG["n_layers"],
).to(DEVICE)

n_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {n_params:,}")
print(model)

train_tensor  = torch.from_numpy(windows_train)
train_loader  = DataLoader(
    TensorDataset(train_tensor),
    batch_size=CFG["batch_size"],
    shuffle=True,
)

optimizer = torch.optim.Adam(model.parameters(), lr=CFG["lr"])
criterion = nn.MSELoss()

## 4 · Training Loop

In [ ]:
train_losses = []
best_loss    = float("inf")
best_path    = MODEL_DIR / "lstm_autoencoder.pt"

for epoch in range(1, CFG["epochs"] + 1):
    model.train()
    epoch_loss = 0.0

    for (batch,) in train_loader:
        batch = batch.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(batch), batch)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        epoch_loss += loss.item() * len(batch)

    epoch_loss /= len(windows_train)
    train_losses.append(epoch_loss)

    if epoch_loss < best_loss:
        best_loss = epoch_loss
        torch.save(model.state_dict(), best_path)

    if epoch % 5 == 0 or epoch == 1:
        print(f"Epoch {epoch:3d}/{CFG['epochs']}  loss={epoch_loss:.6f}")

print(f"\nBest loss: {best_loss:.6f}  → saved to {best_path}")

## 5 · Training Loss Curve

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(range(1, len(train_losses) + 1), train_losses, lw=1.5, color="steelblue")
ax.set_xlabel("Epoch")
ax.set_ylabel("MSE Loss")
ax.set_title("LSTM Autoencoder — Training Loss")
ax.set_yscale("log")
plt.tight_layout()
plt.savefig(PROC_DIR / "10_training_loss.png", bbox_inches="tight")
plt.show()

## 6 · Threshold Calibration

Load the best checkpoint and compute reconstruction errors on the training windows.  
The threshold is set at **μ + 3σ** — ~0.13% false-positive rate under a true Gaussian.

In [ ]:
model.load_state_dict(torch.load(best_path, map_location=DEVICE))

train_errors = collect_errors(model, windows_train, device=DEVICE)
mu, sigma, threshold = compute_threshold(train_errors, k=CFG["k_sigma"])

save_threshold(mu, sigma, threshold, MODEL_DIR / "threshold.json")

print(f"Train errors — mean: {mu:.6f}  std: {sigma:.6f}")
print(f"Threshold (μ+3σ)   : {threshold:.6f}")

# Distribution of training errors
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(train_errors, bins=50, color="steelblue", edgecolor="white", alpha=0.8, density=True)
ax.axvline(threshold, color="red", lw=1.5, ls="--", label=f"Threshold = {threshold:.4f}")
ax.set_xlabel("Reconstruction Error (MSE)")
ax.set_ylabel("Density")
ax.set_title("Training Error Distribution (healthy data only)")
ax.legend()
plt.tight_layout()
plt.savefig(PROC_DIR / "11_train_error_distribution.png", bbox_inches="tight")
plt.show()

## 7 · Anomaly Score on Full Test Set

Run inference on the test windows (the 80% that includes the failure period)  
and plot the reconstruction error over time.

In [ ]:
test_errors = collect_errors(model, windows_test, device=DEVICE)

n_anomalies = (test_errors > threshold).sum()
print(f"Test windows  : {len(test_errors)}")
print(f"Anomalies     : {n_anomalies}  ({100 * n_anomalies / len(test_errors):.1f}%)")

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(test_errors, lw=0.8, color="steelblue", label="Reconstruction error")
ax.axhline(threshold, color="red", lw=1.5, ls="--",
           label=f"Threshold = {threshold:.4f}")
ax.fill_between(
    range(len(test_errors)), test_errors, threshold,
    where=(test_errors > threshold),
    color="red", alpha=0.3, label="Anomaly"
)
ax.set_xlabel("Window index (test set, chronological)")
ax.set_ylabel("Reconstruction MSE")
ax.set_title("Anomaly Score — Test Set (includes bearing failure period)")
ax.legend()
plt.tight_layout()
plt.savefig(PROC_DIR / "12_anomaly_scores_test.png", bbox_inches="tight")
plt.show()

## 8 · Anomaly Score Over Full Run (with timestamps)

In [ ]:
# Reconstruct timestamps for test windows
# Each window represents a position in the feature matrix — load the feature CSV for timestamps
df_features  = pd.read_csv(PROC_DIR / "features_final.csv", index_col="timestamp", parse_dates=True)
TRAIN_FRAC   = 0.20
split_idx    = int(len(df_features) * TRAIN_FRAC)
test_ts      = df_features.index[split_idx:]

# Windows are created with step=1, so window i corresponds to
# the snapshot at index SEQ_LEN-1+i in the test split
window_ts = test_ts[SEQ_LEN - 1 : SEQ_LEN - 1 + len(test_errors)]

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(window_ts, test_errors, lw=0.8, color="steelblue", label="Reconstruction error")
ax.axhline(threshold, color="red", lw=1.5, ls="--",
           label=f"Threshold = {threshold:.4f}")
ax.fill_between(
    window_ts, test_errors, threshold,
    where=(test_errors > threshold),
    color="red", alpha=0.3, label="Anomaly zone"
)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%m-%d"))
plt.xticks(rotation=30)
ax.set_xlabel("Date")
ax.set_ylabel("Reconstruction MSE")
ax.set_title("Bearing Anomaly Score Over Time — LSTM Autoencoder")
ax.legend()
plt.tight_layout()
plt.savefig(PROC_DIR / "13_anomaly_scores_timeline.png", bbox_inches="tight")
plt.show()

## 9 · Detection Lead Time

Estimate how many hours before the final failure the model first triggers an alert.

In [ ]:
anomaly_mask  = test_errors > threshold
first_alert   = anomaly_mask.argmax()          # index of first True
last_window   = len(test_errors) - 1

if anomaly_mask.any():
    first_alert_ts = window_ts[first_alert]
    final_ts       = window_ts[last_window]
    lead_time_h    = (final_ts - first_alert_ts).total_seconds() / 3600

    print(f"First alert   : {first_alert_ts}")
    print(f"Last window   : {final_ts}")
    print(f"Detection lead time : {lead_time_h:.1f} hours")
else:
    print("No anomalies detected — try lowering k_sigma or checking the data.")

## 10 · Isolation Forest Baseline

Compare against a classical anomaly detector trained on the same feature matrix.

In [ ]:
import json
from sklearn.ensemble import IsolationForest

with open(PROC_DIR / "scaler.json") as f:
    scaler = json.load(f)
mu_s    = np.array(scaler["mu"],    dtype=np.float32)
sigma_s = np.array(scaler["sigma"], dtype=np.float32)

df_feat      = pd.read_csv(PROC_DIR / "features_final.csv",
                            index_col="timestamp", parse_dates=True)
X            = ((df_feat.values - mu_s) / sigma_s).astype(np.float32)
X_train_if   = X[:split_idx]
X_test_if    = X[split_idx:]

iso = IsolationForest(n_estimators=200, contamination=0.05, random_state=42)
iso.fit(X_train_if)
# score_samples returns negative anomaly scores; flip sign so higher = more anomalous
if_scores = -iso.score_samples(X_test_if)

fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

axes[0].plot(window_ts, test_errors, lw=0.8, color="steelblue")
axes[0].axhline(threshold, color="red", lw=1.2, ls="--")
axes[0].fill_between(window_ts, test_errors, threshold,
                     where=(test_errors > threshold), color="red", alpha=0.3)
axes[0].set_title("LSTM Autoencoder — Reconstruction Error")
axes[0].set_ylabel("MSE")

if_ts = df_features.index[split_idx:]
axes[1].plot(if_ts, if_scores, lw=0.8, color="darkorange")
if_threshold = np.percentile(if_scores[:int(len(if_scores) * 0.20)], 97)
axes[1].axhline(if_threshold, color="red", lw=1.2, ls="--",
                label=f"97th-pct threshold = {if_threshold:.3f}")
axes[1].fill_between(if_ts, if_scores, if_threshold,
                     where=(if_scores > if_threshold), color="red", alpha=0.3)
axes[1].set_title("Isolation Forest — Anomaly Score")
axes[1].set_ylabel("Score")
axes[1].legend(fontsize=8)
axes[1].xaxis.set_major_formatter(mdates.DateFormatter("%m-%d"))
plt.xticks(rotation=30)

plt.suptitle("LSTM Autoencoder vs Isolation Forest", fontsize=11)
plt.tight_layout()
plt.savefig(PROC_DIR / "14_lstm_vs_isolation_forest.png", bbox_inches="tight")
plt.show()

## 11 · Summary

| Metric | Value |
|---|---|
| Model parameters | see cell 3 |
| Best training loss | see cell 4 |
| Anomaly threshold | μ + 3σ on healthy train errors |
| Anomalies detected (test) | see cell 7 |
| Detection lead time | see cell 9 |
| Saved model | `models/lstm_autoencoder.pt` |
| Saved threshold | `models/threshold.json` |

**Next:** `src/api/main.py` — FastAPI inference endpoint that loads the model and scores live sensor batches.